# Macrocosm photo-z CNN — v4 (Kaggle)
Trains on the **v4** data: `catalog_v4` (cleaned 600k) + **sample_v4.5** (24×24×5 cutouts).
## Notebook settings (right panel)
1. **Accelerator → GPU**, 2. **Internet → On**,
3. **Add Input → Datasets →** add the **`macrocosm-sdss-ugriz-24px-v45`** dataset..


## 1. Setup + data


In [ ]:
import os, glob, shutil
os.environ['CUTOUT_SIZE'] = '24'        # sample_v4.5 is 24x24

# code + v4 splits (needs Internet ON)
![ -d /kaggle/working/CNN-Model ] || git clone -q -b task-v4.5 https://github.com/Le-Wagon-Macrocosm/CNN-Model.git /kaggle/working/CNN-Model
%cd /kaggle/working/CNN-Model
!pip -q install mlflow

# v4.5 dataset: Add Input -> search 'macrocosm-sdss-ugriz-24px-v45' -> add it.
SRC = '/kaggle/input/macrocosm-sdss-ugriz-24px-v45'
assert os.path.isdir(SRC), f'Add the macrocosm-sdss-ugriz-24px-v45 dataset; expected at {SRC}'

# stage a writable data dir: symlink images + copy the catalog AS catalog_v1.parquet (loader's name)
DATA_DIR = '/kaggle/working/data'; os.makedirs(DATA_DIR, exist_ok=True)
for p in glob.glob(f'{SRC}/images_*.npy'):
    d = f'{DATA_DIR}/{os.path.basename(p)}'
    if not os.path.exists(d): os.symlink(p, d)
cat = (glob.glob(f'{SRC}/catalog_v4.parquet') or glob.glob(f'{SRC}/catalog_v1.parquet'))[0]
shutil.copy(cat, f'{DATA_DIR}/catalog_v1.parquet')
print('DATA_DIR =', DATA_DIR, '| shards =', len(glob.glob(f'{DATA_DIR}/images_*.npy')))

import tensorflow as tf
print('GPUs =', tf.config.list_physical_devices('GPU'))

## 2. Train
Uses the v4 `splits/` (train 554,628 / val 45,372). `mlflow_token=None` = train only.


In [ ]:
from photoz_cnn import train

metrics, model = train(
    data_dir=DATA_DIR,
    crop=24,
    preproc='p99',
    N=None,
    batch=256, lr=3e-4, epochs=50,
    run_name='v4-kaggle',
    experiment='oa',
    mlflow_token=None,       # paste the MLflow token to log
)
print(metrics)

## 3. (optional) 3-fold OOF outliers
`cv_outliers.run` writes to `gs://…` (needs GCS auth on Kaggle) — usually skip here.


In [ ]:
# from cv_outliers import run
# df = run(seed=0, data_dir=DATA_DIR, crop=24, preproc='p99')